# Travel Booking Analytics - SQL Analysis

SQL queries executed on the hotel booking dataset using an in-memory SQLite database via Pandas.

### 1. Setup Environment & In-Memory SQLite Database

In [1]:
import sqlite3
import pandas as pd
import numpy as np

# Load dataset and create SQLite table 'bookings'
df = pd.read_csv('hotel_bookings.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('bookings', conn, index=False, if_exists='replace')
print('SQLite database initialized with table: bookings')

SQLite database initialized with table: bookings


### Query 1: Cancellation Rate by Hotel Type

In [2]:
query1 = """
SELECT hotel,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM bookings
GROUP BY hotel;
"""
pd.read_sql(query1, conn)

,hotel,cancellation_rate
0,City Hotel,41.73
1,Resort Hotel,27.76


### Query 2: Cancellation Rate by Customer Type

In [3]:
query2 = """
SELECT customer_type,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM bookings
GROUP BY customer_type
ORDER BY cancellation_rate DESC;
"""
pd.read_sql(query2, conn)

,customer_type,cancellation_rate
0,Transient,40.75
1,Contract,30.96
2,Transient-Party,25.43
3,Group,10.23


### Query 3: Monthly Booking Volume

In [4]:
query3 = """
SELECT arrival_date_month,
    COUNT(*) AS total_bookings
FROM bookings
GROUP BY arrival_date_month
ORDER BY total_bookings DESC;
"""
pd.read_sql(query3, conn)

,arrival_date_month,total_bookings
0,August,13877
1,July,12661
2,May,11791
3,October,11160
4,April,11089
5,June,10939
6,September,10508
7,March,9794
8,February,8068
9,November,6794


### Query 4: High-Value Bookings (Non-Canceled ADR > €200)

In [5]:
query4 = """
SELECT hotel, market_segment, adr 
FROM bookings 
WHERE is_canceled = 0 AND adr > 200
ORDER BY adr DESC 
LIMIT 10;
"""
pd.read_sql(query4, conn)

,hotel,market_segment,adr
0,City Hotel,Offline TA/TO,510.00
1,Resort Hotel,Corporate,508.00
2,City Hotel,Direct,451.50
3,Resort Hotel,Direct,426.25
4,Resort Hotel,Online TA,402.00
5,Resort Hotel,Direct,397.38
6,Resort Hotel,Online TA,392.00
7,Resort Hotel,Online TA,388.00
8,Resort Hotel,Direct,387.00
9,Resort Hotel,Online TA,383.00


### Query 5: Lead Time Category vs Cancellation Rate

In [6]:
query5 = """
SELECT 
    CASE 
        WHEN lead_time <= 30 THEN 'Short' 
        WHEN lead_time <= 90 THEN 'Medium' 
        ELSE 'Long' 
    END AS lead_time_category, 
    COUNT(*) AS bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate 
FROM bookings
GROUP BY lead_time_category
ORDER BY cancellation_rate DESC;
"""
pd.read_sql(query5, conn)

,lead_time_category,bookings,cancellation_rate
0,Long,51131,50.65
1,Medium,29553,37.70
2,Short,38706,18.56
